# PCL Detection — BestModel

This notebook implements a **multitask RoBERTa-base** model for Patronising and Condescending Language (PCL) detection (SemEval 2022 Task 4, Subtask 1).

**Structure:**
1. Imports & Config
2. Data Loading
3. Model Definition (defined once, reused everywhere)
4. Hyperparameter Tuning via **K-Fold Cross-Validation on the train set**
5. Final model training on full train set with best hyperparams
6. Generate `dev.txt` and `test.txt`
7. Error Analysis

> **Why CV?** All hyperparameter decisions are made using only the train set. The official dev set is held out as a true test, ensuring there is no information leakage from hyperparameter selection.

In [17]:
# ── Cell 1: Imports & Global Config ──────────────────────────────────────────
import os, ast, itertools, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import (
    f1_score, precision_recall_fscore_support, confusion_matrix, 
    classification_report, precision_score, recall_score
)
from sklearn.model_selection import StratifiedKFold
from datasets import Dataset
from torch.utils.data import DataLoader
from transformers import (
    AutoConfig, AutoModel, AutoTokenizer,
    DataCollatorWithPadding, Trainer, TrainingArguments, set_seed,
    logging as transformers_logging,
)
from safetensors.torch import load_file
import transformers
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
transformers_logging.set_verbosity_error()

DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_NAME = "roberta-base"
MAX_LEN    = 256
SEED       = 42

print(f"Device: {DEVICE}")

Device: cuda


## 2. Data Loading

Load the raw PCL TSV and merge with the official train/dev split files to obtain binary labels.

In [18]:
# ── Cell 2: Data Loading ──────────────────────────────────────────────────────
def load_data(
    pcl_path   = "Data/dontpatronizeme_pcl.tsv",
    train_path = "Data/train_semeval_parids-labels.csv",
    dev_path   = "Data/dev_semeval_parids-labels.csv",
    test_path  = "Data/task4_test.tsv",
):
    df_pcl = pd.read_csv(
        pcl_path, sep="\t", header=None, skiprows=4,
        names=["par_id","art_id","keyword","country","text","label"],
    ).dropna(subset=["text"])

    def attach_labels(lbl_path):
        lbl = pd.read_csv(lbl_path)
        lbl["cat_labels"] = lbl["label"].apply(ast.literal_eval)
        lbl["y"]          = lbl["cat_labels"].apply(lambda v: int(sum(v) > 0))
        return df_pcl.merge(lbl[["par_id","y","cat_labels"]], on="par_id", how="inner")

    train_df = attach_labels(train_path)
    dev_df   = attach_labels(dev_path)

    test_df = pd.read_csv(
        test_path, sep="\t", header=None,
        names=["par_id","art_id","keyword","country","text"],
    ).dropna(subset=["text"])

    return train_df, dev_df, test_df


train_df, dev_df, test_df = load_data()
print(f"Train: {len(train_df)}  |  Dev: {len(dev_df)}  |  Test: {len(test_df)}")
print(f"Train class balance — PCL: {train_df.y.mean():.2%}  No-PCL: {1-train_df.y.mean():.2%}")

Train: 8375  |  Dev: 2093  |  Test: 3832
Train class balance — PCL: 9.48%  No-PCL: 90.52%


## 3. Model & Helper Definitions

Everything is defined **once** in this cell and reused throughout:

- `MultiTaskPCL` — shared RoBERTa backbone with a binary head (PCL vs No-PCL) and an auxiliary 7-way category head. The auxiliary loss is computed only on positive examples (`pos_only_category_loss=True`) to avoid penalising the model for category predictions on negative instances.
- `tokenize_fn` / `compute_metrics_tuned` — tokenisation and threshold-tuning metric.
- `build_hf_datasets` — converts a DataFrame pair into tokenised HuggingFace Datasets.
- `train_one_fold` — trains a single model instance and returns eval metrics; used for both CV and final training.

In [4]:
# ── Cell 3: Model & Helpers (defined once) ────────────────────────────────────

# ---- Model ----------------------------------------------------------------- #
class MultiTaskPCL(nn.Module):
    """
    RoBERTa backbone with:
      - Binary classification head  (PCL / No-PCL)
      - Auxiliary 7-way multi-label category head
    Total loss = CE(binary, pos_weight) + cat_weight * BCE(categories)
    """
    def __init__(
        self,
        model_name           : str   = "roberta-base",
        hidden_dim           : int   = 256,
        category_loss_weight : float = 0.5,
        pos_only_cat_loss    : bool  = True,
        dropout_p            : float = 0.1,
        pos_weight           : float = 2.3,
    ):
        super().__init__()
        cfg = AutoConfig.from_pretrained(model_name)
        cfg.hidden_dropout_prob          = dropout_p
        cfg.attention_probs_dropout_prob = dropout_p

        self.backbone             = AutoModel.from_pretrained(model_name, config=cfg)
        self.dropout              = nn.Dropout(dropout_p)
        self.projection           = nn.Linear(cfg.hidden_size, hidden_dim)
        self.binary_head          = nn.Linear(hidden_dim, 2)
        self.category_head        = nn.Linear(hidden_dim, 7)

        self.category_loss_weight = category_loss_weight
        self.pos_only_cat_loss    = pos_only_cat_loss
        self.bce                  = nn.BCEWithLogitsLoss()

        w = torch.tensor([1.0, pos_weight])
        self.register_buffer("class_weights", w)

    def _pool(self, hidden, mask):
        # CLS-token pooling
        return hidden[:, 0, :]

    def forward(self, input_ids=None, attention_mask=None, labels=None, cat_labels=None, **kw):
        out    = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self._pool(out.last_hidden_state, attention_mask)
        x      = torch.tanh(self.projection(self.dropout(pooled)))

        bin_logits = self.binary_head(x)
        cat_logits = self.category_head(x)

        loss = None
        if labels is not None:
            ce   = nn.CrossEntropyLoss(weight=self.class_weights.to(bin_logits.device))
            loss = ce(bin_logits, labels)

            if cat_labels is not None and self.category_loss_weight > 0:
                cat_labels_f = cat_labels.float()
                if self.pos_only_cat_loss:
                    mask = labels.bool()
                    if mask.any():
                        loss += self.category_loss_weight * self.bce(
                            cat_logits[mask], cat_labels_f[mask]
                        )
                else:
                    loss += self.category_loss_weight * self.bce(cat_logits, cat_labels_f)

        return {"loss": loss, "logits": bin_logits, "cat_logits": cat_logits}


# ---- Tokenisation ---------------------------------------------------------- #
def make_tokenizer(model_name=MODEL_NAME):
    return AutoTokenizer.from_pretrained(model_name, use_fast=True)

def tokenize_df(df, tokenizer, max_len=MAX_LEN):
    """Convert a DataFrame with 'text', 'y', 'cat_labels' into a tokenised HF Dataset."""
    ds = Dataset.from_pandas(
        df[["text","y","cat_labels"]].rename(columns={"y": "label"})
    )
    return ds.map(
        lambda b: tokenizer(b["text"], truncation=True, max_length=max_len),
        batched=True,
    )


# ---- Metrics --------------------------------------------------------------- #
def _first(x):
    return x[0] if isinstance(x, (tuple, list)) else x

def compute_metrics(eval_pred):
    """Sweep thresholds [0.05, 0.95] on the eval set to maximise F1."""
    logits, labels = eval_pred
    logits, labels = _first(logits), _first(labels)
    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()[:, 1]

    best_f1, best_t = -1.0, 0.5
    for t in np.linspace(0.05, 0.95, 19):
        f1 = f1_score(labels, (probs >= t).astype(int), zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, float(t)

    yhat = (probs >= best_t).astype(int)
    p, r, f1, _ = precision_recall_fscore_support(labels, yhat, average="binary", zero_division=0)
    return {"f1": f1, "precision": p, "recall": r, "best_threshold": best_t}


# ---- Training helper ------------------------------------------------------- #
def train_one_fold(
    tr_ds, val_ds, tokenizer,
    *,
    lr                   = 2e-5,
    num_epochs           = 4,
    batch_size           = 32,
    weight_decay         = 0.02,
    category_loss_weight = 0.5,
    pos_weight           = 2.3,
    dropout_p            = 0.0,
    output_dir           = "./tmp_fold",
    seed                 = SEED,
    save_model           = False,   # True only for the final run
):
    set_seed(seed)
    model = MultiTaskPCL(
        model_name=MODEL_NAME,
        category_loss_weight=category_loss_weight,
        pos_weight=pos_weight,
        dropout_p=dropout_p,
    ).to(DEVICE)

    args = TrainingArguments(
        output_dir              = output_dir,
        num_train_epochs        = num_epochs,
        per_device_train_batch_size = batch_size,
        per_device_eval_batch_size  = 64,
        learning_rate           = lr,
        weight_decay            = weight_decay,
        warmup_ratio            = 0.06,
        lr_scheduler_type       = "linear",
        fp16                    = DEVICE.type == "cuda",
        eval_strategy           = "epoch",
        save_strategy           = "epoch" if save_model else "no",
        load_best_model_at_end  = save_model,
        metric_for_best_model   = "f1",
        dataloader_num_workers  = 2,
        logging_steps           = 50,
        report_to               = "none",
        seed                    = seed,
        log_level               = "error",
    )

    trainer = Trainer(
        model           = model,
        args            = args,
        train_dataset   = tr_ds,
        eval_dataset    = val_ds,
        compute_metrics = compute_metrics,
        data_collator   = DataCollatorWithPadding(tokenizer),
    )
    trainer.remove_callback(transformers.PrinterCallback) 
    trainer.train()
    metrics = trainer.evaluate()
    return trainer, metrics

print("All helpers defined.")

All helpers defined.


## 4. Hyperparameter Tuning via K-Fold Cross-Validation

**Key design choice:** all hyperparameter decisions are made using **stratified k-fold CV on the train set only**. The official dev set is never seen during this stage.

We sweep a small grid of the two most impactful hyperparameters — `learning_rate` and `category_loss_weight` — using 3 folds. For each configuration the mean CV F1 across folds is recorded. The best configuration is then used for the final training run in the next cell.

In [5]:
# ── Cell 4: K-Fold CV Hyperparameter Sweep ────────────────────────────────────
# Adjust the grid as needed. Fewer configs = faster run.
HP_GRID = {
    "lr"                   : [2e-5],
    "category_loss_weight" : [0.5],
    "pos_weight"           : [2.3],       # fixed — from pilot run
    "weight_decay"         : [0.02],      # fixed
    "num_epochs"           : [4],         # fixed
}

N_FOLDS = 5

# Build the full grid list
keys   = list(HP_GRID.keys())
combos = list(itertools.product(*HP_GRID.values()))
print(f"{len(combos)} configs × {N_FOLDS} folds = {len(combos)*N_FOLDS} training runs")

tokenizer = make_tokenizer()
skf       = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
X         = train_df["text"].values
y         = train_df["y"].values

cv_results = []

for combo in combos:
    cfg = dict(zip(keys, combo))
    fold_f1s = []

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
        tr_fold_df  = train_df.iloc[tr_idx].reset_index(drop=True)
        val_fold_df = train_df.iloc[val_idx].reset_index(drop=True)

        tr_ds  = tokenize_df(tr_fold_df,  tokenizer)
        val_ds = tokenize_df(val_fold_df, tokenizer)

        _, metrics = train_one_fold(
            tr_ds, val_ds, tokenizer,
            output_dir=f"./cv_tmp/combo{'_'.join(str(v) for v in combo)}_fold{fold}",
            **cfg,
        )
        fold_f1 = metrics.get("eval_f1", 0.0)
        fold_f1s.append(fold_f1)

    cv_results.append({**cfg, "mean_f1": np.mean(fold_f1s), "std_f1": np.std(fold_f1s)})

cv_df = pd.DataFrame(cv_results).sort_values("mean_f1", ascending=False)
print("\n=== CV Results ===")
display(cv_df)

best_cfg = cv_df.iloc[0].drop(["mean_f1","std_f1"]).to_dict()
best_cfg["lr"]           = float(best_cfg["lr"])
best_cfg["num_epochs"]   = int(best_cfg["num_epochs"])
print("\nBest config:", best_cfg)

1 configs × 5 folds = 5 training runs


Map:   0%|          | 0/6700 [00:00<?, ? examples/s]

Map:   0%|          | 0/1675 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

Map:   0%|          | 0/6700 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/1675 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]


=== CV Results ===


,lr,category_loss_weight,pos_weight,weight_decay,num_epochs,mean_f1,std_f1
0,0.00002,0.5,2.3,0.02,4,0.57982,0.027569



Best config: {'lr': 2e-05, 'category_loss_weight': 0.5, 'pos_weight': 2.3, 'weight_decay': 0.02, 'num_epochs': 4}


## 5. Final Training

Train on the **full train set** using the best hyperparameters from CV. The official dev set is used here only to report the final generalisation performance — not to make any modelling decisions.

In [10]:
# ── Cell 5: Final Training on Full Train Set ──────────────────────────────────
set_seed(SEED)

full_train_ds = tokenize_df(train_df, tokenizer)
dev_ds        = tokenize_df(dev_df,   tokenizer)

final_trainer, final_metrics = train_one_fold(
    full_train_ds, dev_ds, tokenizer,
    output_dir = "./BestModel_beta",
    save_model = False,            # saves best checkpoint to ./BestModel
    **best_cfg,
)

print("\n=== Dev-set metrics (final model) ===")
for k, v in final_metrics.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

THRESH = final_metrics.get("eval_best_threshold", 0.5)
print(f"\nUsing threshold: {THRESH}")

Map:   0%|          | 0/8375 [00:00<?, ? examples/s]

Map:   0%|          | 0/2093 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]


=== Dev-set metrics (final model) ===
  eval_loss: 0.4964
  eval_f1: 0.6221
  eval_precision: 0.6368
  eval_recall: 0.6080
  eval_best_threshold: 0.1000
  eval_runtime: 0.9137
  eval_samples_per_second: 2290.6710
  eval_steps_per_second: 36.1170
  epoch: 4.0000

Using threshold: 0.1


## 6. Generate Predictions

Produce `dev.txt` and `test.txt` in the required format (one `0` or `1` per line).

In [16]:
# ── Cell 6: Generate dev.txt and test.txt ─────────────────────────────────────
collator = DataCollatorWithPadding(tokenizer)

@torch.no_grad()
def predict(df, model, batch_size=64):
    model.eval()
    ds  = Dataset.from_pandas(df[["text"]].copy())
    tok = ds.map(lambda b: tokenizer(b["text"], truncation=True, max_length=MAX_LEN), batched=True)
    tok = tok.remove_columns([c for c in tok.column_names if c not in ["input_ids","attention_mask"]])
    dl  = DataLoader(tok, batch_size=batch_size, shuffle=False, collate_fn=collator)
    probs = []
    for batch in dl:
        batch  = {k: v.to(DEVICE) for k, v in batch.items()}
        logits = model(**batch)["logits"]
        probs.append(torch.softmax(logits, dim=-1)[:, 1].cpu().numpy())
    probs = np.concatenate(probs)
    return probs, (probs >= THRESH).astype(int)


# ── dev.txt: train-only model (honest evaluation) ─────────────────────────────
train_only_model = final_trainer.model
dev_prob, dev_pred = predict(dev_df, model=train_only_model)

dev_f1 = f1_score(dev_df["y"].astype(int), dev_pred)
print(f"Dev F1 (train-only model, threshold={THRESH:.2f}): {dev_f1:.4f}")


# ── test.txt: retrain on train+dev for maximum signal ─────────────────────────
print("\nRetraining on train+dev combined for test predictions...")
combined_ds = tokenize_df(pd.concat([train_df, dev_df], ignore_index=True), tokenizer)
dummy_ds    = tokenize_df(dev_df, tokenizer)   # Trainer requires an eval set

final_trainer_full, _ = train_one_fold(
    combined_ds, dummy_ds, tokenizer,
    output_dir = "./BestModel",
    save_model = True,
    **best_cfg,
)

test_prob, test_pred = predict(test_df, model=final_trainer_full.model)


# ── Write files ───────────────────────────────────────────────────────────────
for path, preds in [("./dev.txt", dev_pred), ("./test.txt", test_pred)]:
    with open(path, "w") as f:
        f.write("\n".join(map(str, preds)) + "\n")
    print(f"Written {path}  ({len(preds)} lines, {int(preds.sum())} positives)")

Map:   0%|          | 0/2093 [00:00<?, ? examples/s]

Dev F1 (train-only model, threshold=0.10): 0.6221

Retraining on train+dev combined for test predictions...


Map:   0%|          | 0/10468 [00:00<?, ? examples/s]

Map:   0%|          | 0/2093 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

Map:   0%|          | 0/3832 [00:00<?, ? examples/s]

Written ./dev.txt  (2093 lines, 190 positives)
Written ./test.txt  (3832 lines, 296 positives)


## 7. Error Analysis

We examine the model's mistakes on the official dev set from two angles:

1. **Global metrics** — confusion matrix, per-class precision/recall.
2. **High-confidence errors** — false positives and false negatives that the model was *most certain* about, to understand systematic failure modes.

In [8]:
# ── Cell 7: Error Analysis ────────────────────────────────────────────────────

y_true = dev_df["y"].astype(int).to_numpy()

# dev_prob and dev_pred already computed in Cell 6 using the train-only model.
# If re-running this cell standalone, uncomment:
# dev_prob, dev_pred = predict(dev_df, model=train_only_model)

# ── 1. Global metrics ─────────────────────────────────────────────────────────
f1   = f1_score(y_true, dev_pred, zero_division=0)
prec = precision_score(y_true, dev_pred, zero_division=0)
rec  = recall_score(y_true, dev_pred, zero_division=0)
cm   = confusion_matrix(y_true, dev_pred)

print(f"Threshold : {THRESH}")
print(f"Precision : {prec:.4f}  |  Recall: {rec:.4f}  |  F1: {f1:.4f}")
print(f"\nConfusion matrix [[TN, FP], [FN, TP]]:\n{cm}")
print(f"\n{classification_report(y_true, dev_pred, digits=4, zero_division=0)}")

# ── 2. Build annotated dev DataFrame ──────────────────────────────────────────
dfA = dev_df.copy()
dfA["p_pos"] = dev_prob
dfA["pred"]  = dev_pred
dfA["true"]  = y_true
dfA["error"] = np.select(
    [(dfA.pred==1)&(dfA.true==0), (dfA.pred==0)&(dfA.true==1),
     (dfA.pred==1)&(dfA.true==1), (dfA.pred==0)&(dfA.true==0)],
    ["FP", "FN", "TP", "TN"]
)

# ── 3. High-confidence errors ─────────────────────────────────────────────────
COLS = ["p_pos", "pred", "true", "keyword", "country", "text"]
pd.set_option("display.max_colwidth", None)

fp = dfA[dfA.error=="FP"].sort_values("p_pos", ascending=False)
fn = dfA[dfA.error=="FN"].sort_values("p_pos", ascending=True)

print("=== High-confidence False Positives ===")
display(fp[COLS].head(12))

print("=== High-confidence False Negatives ===")
display(fn[COLS].head(12))

# ── 4. Threshold curve ────────────────────────────────────────────────────────
curve = []
for t in np.linspace(0.05, 0.95, 19):
    p_t = (dev_prob >= t).astype(int)
    curve.append({
        "threshold" : round(float(t), 2),
        "f1"        : f1_score(y_true, p_t, zero_division=0),
        "precision" : precision_score(y_true, p_t, zero_division=0),
        "recall"    : recall_score(y_true, p_t, zero_division=0),
        "% predicted positive" : float(p_t.mean()),
    })
curve_df = pd.DataFrame(curve).sort_values("f1", ascending=False)

print("=== Threshold sweep (top 10 by F1) ===")
display(
    curve_df.head(10)
    .style.highlight_max(subset=["f1"], color="lightgreen")
    .format({"f1": "{:.4f}", "precision": "{:.4f}", "recall": "{:.4f}", "% predicted positive": "{:.2%}"})
)

# ── 5. Slice analysis: keyword & country ──────────────────────────────────────
def slice_table(df, group_col, min_n=25):
    g = (
        df.groupby(group_col)
        .apply(lambda s: pd.Series({
            "n"         : len(s),
            "pos_rate"  : s["true"].mean(),
            "f1"        : f1_score(s["true"], s["pred"], zero_division=0),
            "precision" : precision_score(s["true"], s["pred"], zero_division=0),
            "recall"    : recall_score(s["true"], s["pred"], zero_division=0),
        }))
        .query(f"n >= {min_n}")
        .sort_values("f1")
    )
    return g

print("=== Hardest keywords (min 25 examples, sorted by F1) ===")
display(slice_table(dfA, "keyword").style.background_gradient(subset=["f1"], cmap="RdYlGn"))

print("=== Hardest countries (min 25 examples, sorted by F1) ===")
display(slice_table(dfA, "country").style.background_gradient(subset=["f1"], cmap="RdYlGn"))

# ── 6. Category-level recall (PCL taxonomy) ───────────────────────────────────
CAT_NAMES = [
    "Unequal treatment", "Shallow solution", "Presupposition",
    "Authority voice", "Metaphor", "Compassion", "The saviour"
]

if "cat_labels" in dfA.columns:
    cat_mat  = np.stack(dfA["cat_labels"].apply(lambda x: np.array(x, dtype=int)))
    pos_mask = dfA["true"].to_numpy() == 1
    fn_mask  = dfA["error"].to_numpy() == "FN"

    rows = []
    for j, cname in enumerate(CAT_NAMES):
        present    = pos_mask & (cat_mat[:, j] == 1)
        n_present  = present.sum()
        if n_present == 0:
            continue
        fn_in_cat  = (present & fn_mask).sum()
        rows.append({
            "category"       : cname,
            "n_positives"    : int(n_present),
            "false_negatives": int(fn_in_cat),
            "recall"         : 1.0 - fn_in_cat / n_present,
        })

    cat_df = pd.DataFrame(rows).sort_values("recall")
    print("=== Per-category recall on positives (lower = model struggles most) ===")
    display(
        cat_df.style
        .background_gradient(subset=["recall"], cmap="RdYlGn")
        .format({"recall": "{:.4f}"})
    )

Threshold : 0.1
Precision : 0.6368  |  Recall: 0.6080  |  F1: 0.6221

Confusion matrix [[TN, FP], [FN, TP]]:
[[1825   69]
 [  78  121]]

              precision    recall  f1-score   support

           0     0.9590    0.9636    0.9613      1894
           1     0.6368    0.6080    0.6221       199

    accuracy                         0.9298      2093
   macro avg     0.7979    0.7858    0.7917      2093
weighted avg     0.9284    0.9298    0.9290      2093

=== High-confidence False Positives ===


,p_pos,pred,true,keyword,country,text
584,0.992640,1,0,in-need,ph,""" I always consider this job as a gift , being a nurse is a reward and task given by God to help those who are in need . Seeing your patient recover from an illness , watching their families smile when you give them care , and hearing the first cry of a newborn are just some of the things that make my work special . It might be a heavy work but it can lighten your heart , "" she expressed ."
1063,0.989928,1,0,hopeless,ph,"But more than taking personal strength from the heroines of Philippine culture , Alma finds greater inspiration in being able to share her skills to build on the dreams of women and children , "" I look at art not as a career but as a spiritual expression . Art should bring out what is innately beautiful , especially to those who are hopeless . """
714,0.989265,1,0,in-need,my,"Jesus begins his teaching in Matthew with the Sermon on the Mount . One group he blesses is those in need of comfort , Blessed are they who mourn , for they will be comforted ( Mt 5:4 ) ."
1129,0.988820,1,0,in-need,in,"Christmas is celebration of the birth of not merely a child , but a child who changed the destiny of humans forever . It is celebration of the fact that God wanted to be a part of the human race and so he took on flesh and blood and became human like us . We can also show unconditional love through our good deeds and helping those who are in need of our help and care . Be human and merciful ."
415,0.986942,1,0,refugee,ke,"E-mail Address : * <h> A clinic called "" Hope "" helps a Syrian refugee boy cope with diabetes"
1686,0.986376,1,0,in-need,za,"Adopt a Mission serves as a platform for the church and for likeminded people to reach out to unemployed families in the communities -- whoever is in need . "" <h> The forgotten people of Brooklyn"
1096,0.984635,1,0,homeless,lk,"A happy day it was indeed when a 31 homeless street dogs found their forever homes in the loving arms of the kind children and adults who visited Embark 's ' Adopt A Dog Day ' at Lyceum International School , Nugegoda on the 15th of September . Students from grades 1 to 8 were invited to attend along with their families and friends . The organizers were delighted by the fact that all the puppies and adult dogs who had been put up for adoption found good homes and new families in record time . In fact , Embark ran out of dogs as the demand was so high , but this was just the first adoption day of many more to come at the Lyceum Schools , so everyone who missed out can find their forever friends at future adoption days ."
971,0.974067,1,0,refugee,us,"In an act of defiance against Hungarian authorities , which had suspended trains to Western Europe , between 1,200 and 2,000 refugees decided to walk from Keleti Station in Budapest to Vienna on Friday . Men , women , children -- even people on crutches -- marched from the late morning to evening , clutching bags filled with all the possessions they had ."
1242,0.973393,1,0,in-need,gb,"She added : "" I am really happy with the awards , but I just want to help those in need . """
634,0.963814,1,0,homeless,ng,"6 years ago she lost her husband -- he died from heart disease , and since then Wood lives alone with her children and a dog . When she learnt about Michael and Cory 's terrible life problems , she realised she had to help . This woman has a big kind heart . There were 2 empty rooms in her house , so she invited the homeless to live there . Many people might consider her action as a madness but Mel says that since her husband 's death , she does n't fear anything ."


=== High-confidence False Negatives ===


,p_pos,pred,true,keyword,country,text
41,0.001008,0,1,vulnerable,ng,"He said : "" I think we can consider introducing a negligible telecom surcharge to be entirely borne by the initiator of a call . In order to protect the poor and vulnerable amongst us , we could structure it to only take effect after the third minute of talk ."
150,0.001026,0,1,disabled,hk,"Cheung said 20 disabled undergraduate students from seven universities will start their eight-week internship in government departments this month , receiving the same salaries as able-bodied colleagues of HK$9,600 a month ."
121,0.001080,0,1,poor-families,us,"The World Health Organization did not give a reason for the increase in deaths , but a provincial health official in Sindh said that the disease hit areas where poor families did not vaccinate their children ."
54,0.001131,0,1,migrant,pk,Famously progressive San Francisco has among America 's most eco-friendly public policies and has declared itself a sanctuary to immigrants it considers persecuted by President Donald Trump 's administration .
117,0.001149,0,1,refugee,my,"IdeaRaya spokesman Maryam Ramli Lee in an interview with Malay Mail Online , September 10 , 2015 . For human rights , Idearaya is shining a spotlight on marginalised communities such as refugees and migrant workers , while its focus for philosophy this year would be on educating children to express themselves philosophically ."
12,0.001176,0,1,homeless,ie,"It 's calculated that over 204,000 days of purpose-built residential accommodation to otherwise potentially homeless elderly men and women have been delivered by this personally driven altruistic act alone ."
142,0.001183,0,1,poor-families,au,"Michael Gove 's recent suggestion that inadequate financial management skills among poor families are to blame for the increasing demand on food banks has , unsurprisingly , sparked an angry response . Critics feel the Conservative Party is out of touch with the pressures on ordinary families ."
108,0.001190,0,1,hopeless,bd,"Calling for an immediate political solution to resolve the crisis in Syria , Erdogan said : "" Turkey 's incursion into northern Syria in early September had led to establishing peace , balance and stability in a region taken over by hopelessness "" ."
1965,0.001204,0,1,poor-families,hk,"A top health official said today that the government could consider subsidies to help poor families pay for healthy food -- or imposing taxes on unhealthy products -- if other efforts fail to encourage better eating habits among Hong Kong residents . Dr Regina Ching from the Health Department said such moves could be explored as a way to cut levels of chronic illnesses in the city , such ..."
42,0.001238,0,1,hopeless,pk,"The Global Gender Gap Report 2016 has ranked Pakistan 143 out of 144 countries . This really demonstrates the failure of feminists who are hopeless to break the historical shackles of bondage on women and establish favorable political , educational , economic , and health conditions ."


=== Threshold sweep (top 10 by F1) ===


...

,n,pos_rate,f1,precision,recall
keyword,,,,,
women,233.000000,0.060086,0.363636,0.500000,0.285714
immigrant,218.000000,0.032110,0.444444,1.000000,0.285714
disabled,194.000000,0.072165,0.521739,0.666667,0.428571
migrant,206.000000,0.024272,0.571429,1.000000,0.400000
vulnerable,209.000000,0.095694,0.585366,0.571429,0.600000
hopeless,217.000000,0.119816,0.588235,0.600000,0.576923
poor-families,190.000000,0.200000,0.594595,0.611111,0.578947
refugee,188.000000,0.069149,0.615385,0.615385,0.615385
homeless,212.000000,0.136792,0.666667,0.645161,0.689655


=== Hardest countries (min 25 examples, sorted by F1) ===


,n,pos_rate,f1,precision,recall
country,,,,,
au,98.000000,0.051020,0.428571,0.333333,0.600000
bd,103.000000,0.067961,0.428571,0.428571,0.428571
ph,105.000000,0.142857,0.428571,0.461538,0.400000
lk,85.000000,0.105882,0.444444,0.444444,0.444444
tz,94.000000,0.117021,0.470588,0.666667,0.363636
us,114.000000,0.087719,0.500000,0.500000,0.500000
in,104.000000,0.067308,0.500000,0.600000,0.428571
za,110.000000,0.100000,0.545455,0.545455,0.545455
gb,127.000000,0.070866,0.588235,0.625000,0.555556


=== Per-category recall on positives (lower = model struggles most) ===


,category,n_positives,false_negatives,recall
6,The saviour,11,6,0.4545
2,Presupposition,62,26,0.5806
3,Authority voice,38,14,0.6316
5,Compassion,106,36,0.6604
0,Unequal treatment,142,45,0.6831
4,Metaphor,52,16,0.6923
1,Shallow solution,36,11,0.6944


In [11]:
# ── Cell 8: High-Confidence False Positives & False Negatives ─────────────────
pd.set_option("display.max_colwidth", None)
N    = 5
cols = ["p_pos", "pred", "true", "keyword", "country", "text"]

fp = dfA[dfA.error=="FP"].sort_values("p_pos", ascending=False).head(N)
fn = dfA[dfA.error=="FN"].sort_values("p_pos", ascending=True).head(N)

print("=" * 80)
print(f"Top {N} HIGH-CONFIDENCE FALSE POSITIVES")
print("=" * 80)
for i, row in fp[cols].reset_index(drop=True).iterrows():
    print(f"\n[FP #{i+1}] p_pos={row.p_pos:.4f}  keyword={row.keyword}  country={row.country}")
    print(row.text)

print("\n" + "=" * 80)
print(f"Top {N} HIGH-CONFIDENCE FALSE NEGATIVES")
print("=" * 80)
for i, row in fn[cols].reset_index(drop=True).iterrows():
    print(f"\n[FN #{i+1}] p_pos={row.p_pos:.4f}  keyword={row.keyword}  country={row.country}")
    print(row.text)

Top 5 HIGH-CONFIDENCE FALSE POSITIVES

[FP #1] p_pos=0.9926  keyword=in-need  country=ph
" I always consider this job as a gift , being a nurse is a reward and task given by God to help those who are in need . Seeing your patient recover from an illness , watching their families smile when you give them care , and hearing the first cry of a newborn are just some of the things that make my work special . It might be a heavy work but it can lighten your heart , " she expressed .

[FP #2] p_pos=0.9899  keyword=hopeless  country=ph
But more than taking personal strength from the heroines of Philippine culture , Alma finds greater inspiration in being able to share her skills to build on the dreams of women and children , " I look at art not as a career but as a spiritual expression . Art should bring out what is innately beautiful , especially to those who are hopeless . "

[FP #3] p_pos=0.9893  keyword=in-need  country=my
Jesus begins his teaching in Matthew with the Sermon on the Mount .